# Day 7 — Solution: Beta Lab

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 5)
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

from qrc.data import get_prices
from qrc.synth import synthetic_returns
from qrc.universe import load_universe

if DATA_SOURCE == "real":
    names = load_universe("research50")[:10]
    px = get_prices(names + ["SPY"], start="2015-01-01")
    rets = np.log(px).diff().dropna(how="all")
    rets = rets.dropna()  # 15y of megacaps: dropna per full-panel is fine; stated for the report
else:
    rets = synthetic_returns(n_days=2500, n_assets=11, seed=31, corr=0.45,
                             drift_spread=0.00012)
    rets.columns = ["SPY"] + [f"S{i}" for i in range(1, 11)]
print(f"panel: {rets.shape[0]} days x {rets.shape[1]} series "
      f"({rets.index[0].date()} -> {rets.index[-1].date()})")

MKT = rets["SPY"]
NAMES = [c for c in rets.columns if c != "SPY"]

## Part 1 — the table

In [ ]:
def ols_fit(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    s2 = e @ e / (n - 2)
    se_c = np.sqrt(np.diag(s2 * XtX_inv))
    meat = X.T @ (e[:, None] ** 2 * X)
    se_w = np.sqrt(np.diag(XtX_inv @ meat @ XtX_inv)) * np.sqrt(n / (n - 2))
    r2 = 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))
    return b, se_c, se_w, r2

rows = {}
for t in NAMES:
    d = pd.concat([MKT, rets[t]], axis=1).dropna()
    b, se_c, se_w, r2 = ols_fit(d.iloc[:, 0].values, d.iloc[:, 1].values)
    rows[t] = [b[1], se_c[1], se_w[1], b[1]/se_w[1], b[0]*252*100, se_w[0]*252*100, r2, se_w[1]/se_c[1]]
tab = pd.DataFrame(rows, index=["beta", "SE_class", "SE_white", "t_white",
                                "alpha_pct_yr", "SE_alpha_pct_yr", "R2", "SE_ratio"]).T
tab = tab.sort_values("beta", ascending=False)
print(tab.round(3).to_string())
print("\nSE-ratio outliers (>20% from 1):", tab.index[(tab.SE_ratio - 1).abs() > 0.2].tolist())

**Expected pattern (real mode).** β̂ spreads ~0.7–1.8 across the ten
(TSLA/NVDA high, KO/PG/XLP-types low); all t(White) massive (slopes are
pinned down at this n — significance is NOT the beta table's question);
α̂/yr mostly within ±2 SE of zero with a couple of famous exceptions — the
honest scan sentence is "α̂ is generally indistinguishable from zero at
name level; the two apparent exceptions are within multiple-testing
expectations for ten looks." R² 0.15–0.55. SE-ratio outliers flag names
whose residual variance loads on market magnitude — **if your favorite
name sits there, its classical-SE literature is overstated.**

**Synthetic mode.** Same code runs; planted ordering (drift_spread by
column, shared factor correlation) is recovered in the β̂ ranking — the
unit test passes if the rank order is monotone; if it isn't, find the bug
before Part 2.

## Part 2 — rolling betas

In [ ]:
W = 252
roll_b = rets[NAMES].rolling(W).cov(MKT).div(MKT.rolling(W).var(), axis=0)
roll_b = roll_b.dropna()

ax = roll_b.plot(lw=0.9, alpha=0.85)
# SE band for the wiggliest single name (closed form, no loop)
name = roll_b.std().idxmax()
rho = rets[name].rolling(W).corr(MKT)
sig_e = rets[name].rolling(W).std() * np.sqrt(1 - rho**2)
se_roll = (sig_e / (MKT.rolling(W).std() * np.sqrt(W))).reindex(roll_b.index)
ax.fill_between(roll_b.index, (roll_b[name] - 2*se_roll).values,
                (roll_b[name] + 2*se_roll).values, alpha=0.25)
ax.axhline(1, color="k", lw=0.6, ls="--")
ax.set_title(f"252-day rolling betas; band = ±2SE for {name}")
plt.tight_layout(); plt.show()
print(f"wiggliest name: {name}  rolling beta range {roll_b[name].min():.2f}..{roll_b[name].max():.2f}")

**Expected pattern.** All ten lines **wander** — 0.6 band swings over a
decade for single names are routine; 2020 shows a synchronized lurch
(betas to SPY moved as correlations jumped — the market factor's share of
variance rose; recall from module 03 that correlations spike in crises).
The ±2SE band covers a *large* fraction of the wiggle — visual proof that
much "beta drift" is sampling noise in a trench coat. **The band is the
honest part of the chart; prefer it to any smooth story about
time-varying beta.** (Day 17 makes that check quantitative.)

## Part 3 — stability

In [ ]:
rng_lo_hi = roll_b.quantile([0.05, 0.95]).T
print("5-95% rolling range vs full-sample SE:")
cmp = pd.concat([rng_lo_hi, tab["SE_class"]], axis=1)
cmp.columns = ["p05", "p95", "SE_full"]
cmp["range"] = cmp.p95 - cmp.p05
cmp["range_over_2SE"] = cmp.range / (2 * cmp.SE_full)
print(cmp.round(2).to_string())

by_year = {yr: rets.loc[str(yr)][NAMES].apply(lambda s: np.cov(s, MKT.loc[str(yr)])[0,1] / MKT.loc[str(yr)].var())
           for yr in sorted(rets.index.year.unique())}
by_year = pd.DataFrame(by_year)   # rows: names, cols: years
yrs = by_year.columns
yoy = np.mean([by_year[yrs[i]].corr(by_year[yrs[i+1]]) for i in range(len(yrs)-1)])
print(f"\nYoY cross-sectional beta persistence (avg corr): {yoy:+.2f}")

# null benchmark: SAME pipeline on a world with constant planted betas
from qrc.synth import synthetic_returns
rng_pipe = synthetic_returns(n_days=len(rets), n_assets=11, seed=77, corr=0.45)
wiggle = rng_pipe.rolling(W).cov(rng_pipe.iloc[:,0]).div(rng_pipe.iloc[:,0].rolling(W).var(), axis=0).dropna()
print(f"constant-beta world: 5-95% range of rolling estimate {wiggle.quantile(0.95).mean()-wiggle.quantile(0.05).mean():.2f} "
      f"<-- how far a CONSTANT beta wanders through this window")

**Expected pattern.** The 5–95% range per name ≈ 3–6× the full-sample
2·SE band — but the overlapping windows share 251/252 of their data, so
the naive ratio overstates true time-variation; the **constant-beta null
benchmark** shows the same pipeline wandering ~0.4–0.6 with NO time
variation at all: much of the roll is mechanical. YoY persistence lands
~+0.5 to +0.8 real / high 0.9s synthetic — **beta rankings persist, levels
drift.** The usable sentence for module 07: "rank stocks by beta with
confidence; quote any single name's beta with a band."

## Part 4 — the audit (exemplar)

Benchmark: SPY — a US-large-cap lens; against it, sector tilts read as
beta (an oil name's "beta" is partly oil's correlation to the index);
a different benchmark (broad market, sector-neutral) moves every number.
Window: 2015→ covers one low-vol bull, one crash, one hiking cycle — all
estimates are this menu's average; rolling estimates re-answer per regime.
Adjustment: adjusted prices — split/dividend-consistent returns (module
05's machine-checked construction); raw closes would inject phantom moves
into every β̂ around ex-dates. Universe: research50 = *today's* 50
survivors — no delisted names, no small caps, so the cross-section's
average beta is conditioned on survival and size; nothing here transfers
to the small-cap universe where beta dispersion is wider. **These betas
cannot be used for: precise hedge ratios on any single name (band first),
for claims about the population of stocks (survivorship), or as stable
inputs to a forward-looking risk model without a shrinkage/rolling plan.**
```

## What a completed lab proves

You can produce a beta table a desk would accept, chart its time
behavior, *and* quantify why the point estimates spend most of their
credibility the moment the window moves. **The artifact is the report;
the skill is the third column (SE) being non-optional.**